<a href="https://colab.research.google.com/github/Katyayini-Sharma/Celebal-Internship-2026/blob/main/week7_Katyayini_Sharma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Document Question Answering System (RAG)

We build an end-to-end **Retrieval-Augmented Generation (RAG)** system that answers questions grounded in a custom document, instead of relying only on a language model's internal (and possibly outdated or hallucinated) knowledge.

**Pipeline stages**
1. Document Ingestion
2. Text Chunking
3. Embedding Creation
4. Vector Database (FAISS)
5. Query Processing
6. Context Retrieval
7. Answer Generation
8. Experiments: Hybrid Search (keywoed + vector) and RE-ranking

**Custom dataset used:** a knowledge-base document summarizing the core deep learning architectures covered earlier in this internship (Autoencoders, CNN vs ANN, RNN/LSTM/GRU); this keeps the RAG demo grounded in domain-specific content rather than a generic public dataset.

# 1. Setup & Installation

In [1]:
!pip install -q sentence-transformers faiss-cpu transformers torch pypdf rank_bm25

In [2]:
import os, time, socket, textwrap
import numpy as np

print("Environment ready.")

Environment ready.


# 2. Backend Auto-Detection

This is designed to run on a platform which has full internet access, so it uses:
- `sentence-transformers` (`all-MiniLM-L6-v2`) for embeddings
- `google/flan-t5-base` (via `transformers`) for answer generation

Some sandboxed / offline environments cannot reach the Hugging Face Hub. The cell
below auto-detects connectivity and transparently falls back to:
- **TF-IDF** vectors (scikit-learn) for embeddings
- An **extractive** generator (returns the most relevant retrieved sentence(s))

so the pipeline always runs end-to-end and produces real, grounded answers either way.

In [3]:
def host_reachable(url="https://huggingface.co", timeout=4):
    try:
        import urllib.request
        req = urllib.request.Request(url, method="HEAD")
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return resp.status < 400
    except Exception:
        return False

HF_AVAILABLE = host_reachable()
print(f"Hugging Face Hub reachable: {HF_AVAILABLE}")
print("Embedding backend:", "sentence-transformers (MiniLM-L6-v2)" if HF_AVAILABLE else "TF-IDF (scikit-learn fallback)")
print("Generation backend:", "flan-t5-base" if HF_AVAILABLE else "extractive fallback (retrieved-context based)")


Hugging Face Hub reachable: True
Embedding backend: sentence-transformers (MiniLM-L6-v2)
Generation backend: flan-t5-base


# 3. Documenet Ingestion

Loads PDFs or plain text files into a single raw-text string.
Tries Google Drive first (for Colab); if Drive isn't mounted or the file isn't found, falls back to a small built-in smaple knowledge base so the notebook still runs end-to-end standalone (e.g. for grading outside Colab).

In [4]:
DRIVE_AVAILABLE = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_AVAILABLE = True
except Exception as e:
    print(f"Google Drive not available ({e.__class__.__name__}); will use local fallback.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
if DRIVE_AVAILABLE:
    print(os.listdir('/content/drive/MyDrive'))


['Classroom', 'Untitled presentation (1).gslides', 'Copy of Elegant Education Pack for Students Pink by Slidesgo.gslides', 'Copy of Tourist Attractions Social Media Strategy by Slidesgo (1).gslides', 'Copy of Tourist Attractions Social Media Strategy by Slidesgo.gslides', 'The Power Of Tourism.gslides', 'Untitled presentation.gslides', 'Aadhar Card Updated.jpeg', 'Resume', '14.02.25', 'LNMIIT PPT', 'ConstiTUT PPT LNMIIT', 'IMG-20250611-WA0049.jpg', 'IMG-20250915-WA0007.jpg', 'Colab Notebooks', 'Documents']


In [6]:
if DRIVE_AVAILABLE:
    print(os.listdir('/content/drive/My Drive/Colab Notebooks/Week 7'))


['week7_Katyayini_Sharma.ipynb', 'knowledge_base (1).txt', 'knowledge_base.txt']


In [7]:
from pypdf import PdfReader

def load_document(path: str) -> str:
    """Load a .pdf or .txt file and return its raw text."""
    if path.lower().endswith(".pdf"):
        reader = PdfReader(path)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

# Small embedded fallback so the notebook is self-contained when Drive isn't available
FALLBACK_DOC = """Denoising autoencoders are neural networks trained to reconstruct clean data from corrupted or noisy input. They consist of an encoder that compresses the input into a latent representation and a decoder that reconstructs the original signal, evaluated typically using reconstruction loss such as MSE and metrics like PSNR.

Convolutional Neural Networks differ from standard Artificial Neural Networks in image classification by using convolutional filters that exploit spatial locality and parameter sharing, making them far more efficient and accurate on image data than fully connected ANNs which flatten pixel grids and lose spatial structure.

Long Short-Term Memory networks solve the vanishing gradient problem present in vanilla RNNs by introducing gating mechanisms: an input gate, forget gate, and output gate, along with a cell state that allows gradients to flow over long sequences without decaying.

Gated Recurrent Units are a simplified variant of LSTM that combine the forget and input gates into a single update gate and merge the cell state with the hidden state, resulting in fewer parameters and faster training while achieving comparable performance on many sequence tasks.

A Retrieval-Augmented Generation pipeline typically consists of document ingestion, text chunking, embedding creation, vector storage, query embedding, context retrieval, and grounded answer generation using a language model conditioned on the retrieved context."""

DOC_PATH = "/content/drive/My Drive/Colab Notebooks/Week 7/knowledge_base.txt"

if DRIVE_AVAILABLE and os.path.exists(DOC_PATH):
    raw_text = load_document(DOC_PATH)
    doc_source = DOC_PATH
else:
    raw_text = FALLBACK_DOC
    doc_source = "built-in fallback sample (Drive/file unavailable)"

print(f"Loaded document: {doc_source}")
print(f"Total characters: {len(raw_text)}")
print(f"Total words: {len(raw_text.split())}")
print("\n--- Preview ---\n")
print(raw_text[:400], "...")


Loaded document: /content/drive/My Drive/Colab Notebooks/Week 7/knowledge_base.txt
Total characters: 3996
Total words: 571

--- Preview ---

Deep Learning Internship Notes: Core Architectures

Autoencoders and Denoising Autoencoders
An autoencoder is a neural network trained to reconstruct its own input. It consists of an encoder, which compresses the input into a smaller latent representation, and a decoder, which reconstructs the original input from that representation. A denoising autoencoder is a variant that is trained on corrupte ...


# 4. Text Chunking

Splits the raw text into overlapping word-level chunks. Overlap preserves context across chunk boundaries so an answer that spans two chunks isn't lost.

In [8]:
def chunk_text(text: str, chunk_size: int = 80, overlap: int = 20):
    """Sliding-window chunking over words."""
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        chunk_words = words[start:start + chunk_size]
        if not chunk_words:
            break
        chunks.append(" ".join(chunk_words))
        if start + chunk_size >= len(words):
            break
    return chunks

CHUNK_SIZE = 80
CHUNK_OVERLAP = 20
chunks = chunk_text(raw_text, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"Chunk size (words): {CHUNK_SIZE}")
print(f"Chunk overlap (words): {CHUNK_OVERLAP}")
print(f"Number of chunks created: {len(chunks)}")
print("\n--- Sample chunk (#0) ---\n")
print(chunks[0])
print("\n--- Sample chunk (#1) ---\n")
print(chunks[1])


Chunk size (words): 80
Chunk overlap (words): 20
Number of chunks created: 10

--- Sample chunk (#0) ---

Deep Learning Internship Notes: Core Architectures Autoencoders and Denoising Autoencoders An autoencoder is a neural network trained to reconstruct its own input. It consists of an encoder, which compresses the input into a smaller latent representation, and a decoder, which reconstructs the original input from that representation. A denoising autoencoder is a variant that is trained on corrupted (noisy) input images, but the target output remains the clean original image. This forces the network to learn robust features that capture

--- Sample chunk (#1) ---

images, but the target output remains the clean original image. This forces the network to learn robust features that capture the true underlying structure of the data rather than memorizing noise. Denoising autoencoders are commonly evaluated using Mean Squared Error (MSE), which measures the average squared difference bet

# 5. Embedding Creation

Converts each text chunk into a numeric vector capturing its semantic meaning.

In [9]:
EMBED_DIM = None
embed_fn = None
tfidf_vectorizer = None

if HF_AVAILABLE:
    from sentence_transformers import SentenceTransformer
    st_model = SentenceTransformer("all-MiniLM-L6-v2")
    EMBED_DIM = st_model.get_sentence_embedding_dimension()

    def embed_fn(texts):
        return st_model.encode(texts, normalize_embeddings=True).astype("float32")
else:
    from sklearn.feature_extraction.text import TfidfVectorizer
    tfidf_vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_vectorizer.fit(chunks)
    EMBED_DIM = len(tfidf_vectorizer.vocabulary_)

    def embed_fn(texts):
        vecs = tfidf_vectorizer.transform(texts).toarray().astype("float32")
        norms = np.linalg.norm(vecs, axis=1, keepdims=True)
        norms[norms == 0] = 1
        return vecs / norms

chunk_embeddings = embed_fn(chunks)
print(f"Embedding dimension: {EMBED_DIM}")
print(f"Embedding matrix shape: {chunk_embeddings.shape}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_10621/3780689557.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBED_DIM = st_model.get_sentence_embedding_dimension()


Embedding dimension: 384
Embedding matrix shape: (10, 384)


# 6. Vector Database

Stores embeddings in a FAISS index for fast similarity search (cosine similarity via normalized inner product.)

In [10]:
import faiss

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)

print(f"Vector store: FAISS IndexFlatIP")
print(f"Vectors stored: {index.ntotal}")
print(f"Vector dimensionality: {index.d}")


Vector store: FAISS IndexFlatIP
Vectors stored: 10
Vector dimensionality: 384


# 7 & 8. Query Processing & Context Retrieval

Embeds the user's question and retrieves the top-k most similar chunks from the vector store.

In [11]:
def retrieve(query: str, k: int = 3):
    query_vec = embed_fn([query])
    scores, idxs = index.search(query_vec, k)
    results = [
        {"chunk": chunks[i], "score": float(scores[0][rank])}
        for rank, i in enumerate(idxs[0]) if i != -1
    ]
    return results

# quick sanity check
sample_results = retrieve("What is a denoising autoencoder?", k=2)
for r in sample_results:
    print(f"score={r['score']:.4f}  chunk={r['chunk'][:120]}...")


score=0.7721  chunk=Deep Learning Internship Notes: Core Architectures Autoencoders and Denoising Autoencoders An autoencoder is a neural ne...
score=0.6455  chunk=images, but the target output remains the clean original image. This forces the network to learn robust features that ca...


# 9. Answer Generation

Feeds the retrieved context plus the original question into a language model to produce a grounded answer.

In [12]:
if HF_AVAILABLE:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    t5_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

    def generate_answer(query: str, context_chunks):
        context = "\n".join(c["chunk"] for c in context_chunks)
        prompt = f"Answer the question using only the context below.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
        inputs = t5_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        outputs = t5_model.generate(**inputs, max_new_tokens=100)
        return t5_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
else:
    def generate_answer(query: str, context_chunks):
        best_chunk = context_chunks[0]["chunk"] if context_chunks else ""
        sentences = [s.strip() for s in best_chunk.split(".") if s.strip()]
        return (". ".join(sentences[:2]) + ".") if sentences else "No relevant context found."

print("Answer generator ready.")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Answer generator ready.


# 10. End-to-End RAG Pipeline

In [13]:
def ask(query: str, k: int = 3, verbose: bool = True):
    t0 = time.time()
    retrieved = retrieve(query, k=k)
    t1 = time.time()
    answer = generate_answer(query, retrieved)
    t2 = time.time()

    if verbose:
        print(f"Q: {query}")
        print(f"A: {answer}")
        print(f"  [retrieval: {t1 - t0:.3f}s | generation: {t2 - t1:.3f}s | chunks used: {len(retrieved)}]")
        for r in retrieved:
            print(f"    - score={r['score']:.4f}  \"{r['chunk'][:90]}...\"")
        print()
    return {"query": query, "answer": answer, "retrieved": retrieved,
            "retrieval_time": t1 - t0, "generation_time": t2 - t1}

result = ask("What is the main idea of the document?")


Q: What is the main idea of the document?
A: purely on the parameters of a language model, a RAG system first retrieves relevant chunks of text from an external knowledge base using vector similarity search, and then provides those chunks as context to a language model so that it can generate an answer grounded in real data
  [retrieval: 0.041s | generation: 11.234s | chunks used: 3]
    - score=0.2833  "purely on the parameters of a language model, a RAG system first retrieves relevant chunks..."
    - score=0.2069  "documents that the language model was never trained on. The core pipeline consists of docu..."
    - score=0.1125  "for reconstruction tasks like autoencoders. Early stopping, which monitors validation loss..."



# 11. Validation: Testing with Dynamic Sample Questions

Grounded, context-aware answers for a set of domain-specific test questions, with retrieval scores logged for validation.

In [14]:
test_questions = [
    "What is a denoising autoencoder and how is it evaluated?",
    "How does a CNN differ from an ANN for image classification?",
    "What problem do LSTMs solve compared to vanilla RNNs?",
    "How is a GRU different from an LSTM?",
    "What are the main stages of a RAG pipeline?",
    "What optimizer and loss functions were used during training?",
]

validation_log = [ask(q) for q in test_questions]


Q: What is a denoising autoencoder and how is it evaluated?
A: A denoising autoencoder is a variant that is trained on corrupted (noisy) input images, but the target output remains the clean original image
  [retrieval: 0.090s | generation: 7.988s | chunks used: 3]
    - score=0.7454  "Deep Learning Internship Notes: Core Architectures Autoencoders and Denoising Autoencoders..."
    - score=0.6078  "images, but the target output remains the clean original image. This forces the network to..."
    - score=0.4293  "quality in decibels, with higher values indicating better reconstruction. Convolutional Ne..."

Q: How does a CNN differ from an ANN for image classification?
A: CNNs generally outperform ANNs on image classification tasks such as CIFAR-10 because they require fewer parameters and generalize better to unseen images
  [retrieval: 0.068s | generation: 4.785s | chunks used: 3]
    - score=0.6479  "quality in decibels, with higher values indicating better reconstruction. Convoluti

# 12. Improvements & Experiments

## 12.1 Hybrid Search (Keyword & Vector)

Combines BM25 keyword-matching scores with vector similarity scores to catch cases where exact terminology matters (e.g. acronyms like "GRU" or "PSNR") that pure semantic search can sometimes under-weight.

In [15]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [c.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

def hybrid_retrieve(query: str, k: int = 3, alpha: float = 0.5):
    """alpha weights vector similarity vs. BM25 keyword score (0=pure BM25, 1=pure vector)."""
    vec_scores = (embed_fn([query]) @ chunk_embeddings.T)[0]
    bm25_scores = np.array(bm25.get_scores(query.lower().split()))

    def normalize(arr):
        rng = arr.max() - arr.min()
        return (arr - arr.min()) / rng if rng > 0 else np.zeros_like(arr)

    combined = alpha * normalize(vec_scores) + (1 - alpha) * normalize(bm25_scores)
    top_idx = np.argsort(combined)[::-1][:k]
    return [{"chunk": chunks[i], "score": float(combined[i])} for i in top_idx]

hybrid_result = hybrid_retrieve("What does GRU stand for and simplify?", k=3)
for r in hybrid_result:
    print(f"score={r['score']:.4f}  chunk={r['chunk'][:110]}...")


score=0.6909  chunk=such as random rotations, flips, and shifts, can further improve a CNN's ability to generalize by artificially...
score=0.6768  chunk=predictions. Vanilla RNNs suffer from the vanishing gradient problem, which makes it difficult for them to lea...
score=0.4266  chunk=documents that the language model was never trained on. The core pipeline consists of document ingestion, text...


## 12.2 Simple Re-ranking

Re-ranks the vector-retrieved candidates by lexical overlap with the query terms, approximating what a dedicated cross-encoder re-ranker would do, without requiring an additional model download.

In [16]:
def rerank(query: str, candidates, top_n: int = 3):
    query_terms = set(query.lower().split())

    def overlap_score(chunk):
        chunk_terms = set(chunk.lower().split())
        return len(query_terms & chunk_terms) / max(len(query_terms), 1)

    reranked = sorted(
        candidates,
        key=lambda c: (0.6 * c["score"] + 0.4 * overlap_score(c["chunk"])),
        reverse=True,
    )
    return reranked[:top_n]

candidates = retrieve("What is Retrieval-Augmented Generation?", k=5)
reranked = rerank("What is Retrieval-Augmented Generation?", candidates, top_n=3)
for r in reranked:
    print(f"score={r['score']:.4f}  chunk={r['chunk'][:110]}...")


score=0.6373  chunk=are a simplified alternative to LSTM that combine the forget and input gates into a single update gate and mer...
score=0.2288  chunk=Deep Learning Internship Notes: Core Architectures Autoencoders and Denoising Autoencoders An autoencoder is a...
score=0.3363  chunk=documents that the language model was never trained on. The core pipeline consists of document ingestion, text...


## 12.3 Chunking Strategy Comparison

Compares retrieval behaviour under a smaller vs. larger chunk size.

In [17]:
def build_index(chunk_size, overlap):
    local_chunks = chunk_text(raw_text, chunk_size, overlap)
    local_embeddings = embed_fn(local_chunks)
    local_index = faiss.IndexFlatIP(local_embeddings.shape[1])
    local_index.add(local_embeddings)
    return local_chunks, local_embeddings, local_index

configs = [(40, 10), (80, 20), (150, 30)]
for size, ov in configs:
    c, e, idx = build_index(size, ov)
    print(f"chunk_size={size:<4} overlap={ov:<4} -> {len(c):>3} chunks | index size={idx.ntotal}")


chunk_size=40   overlap=10   ->  19 chunks | index size=19
chunk_size=80   overlap=20   ->  10 chunks | index size=10
chunk_size=150  overlap=30   ->   5 chunks | index size=5


# 13. System Metrics Report

Summary of the configuration used for this run, for submission documentation.

In [18]:
metrics_report = f"""
DOCUMENT QA (RAG) - SYSTEM METRICS REPORT
==========================================
Document source          : {DOC_PATH}
Document size             : {len(raw_text.split())} words / {len(raw_text)} characters

Chunking
  Strategy                : sliding window, word-based
  Chunk size               : {CHUNK_SIZE} words
  Chunk overlap             : {CHUNK_OVERLAP} words
  Total chunks              : {len(chunks)}

Embeddings
  Backend                  : {"sentence-transformers (all-MiniLM-L6-v2)" if HF_AVAILABLE else "TF-IDF (scikit-learn)"}
  Embedding dimension        : {EMBED_DIM}

Vector Store
  Engine                   : FAISS
  Index type                : IndexFlatIP (cosine via normalized inner product)
  Vectors stored             : {index.ntotal}

Generation
  Backend                  : {"google/flan-t5-base (transformers pipeline)" if HF_AVAILABLE else "extractive fallback (top-context sentence extraction)"}

Experiments
  Hybrid search              : BM25 + vector similarity, weighted (alpha=0.5)
  Re-ranking                 : lexical-overlap re-scoring of top-5 candidates
  Chunking variants tested    : (40,10), (80,20), (150,30)

Validation
  Test questions run          : {len(test_questions)}
  Avg retrieval time (s)      : {np.mean([r['retrieval_time'] for r in validation_log]):.4f}
  Avg generation time (s)     : {np.mean([r['generation_time'] for r in validation_log]):.4f}
"""
print(metrics_report)

with open("metrics_report.txt", "w") as f:
    f.write(metrics_report)



DOCUMENT QA (RAG) - SYSTEM METRICS REPORT
Document source          : /content/drive/My Drive/Colab Notebooks/Week 7/knowledge_base.txt
Document size             : 571 words / 3996 characters

Chunking
  Strategy                : sliding window, word-based
  Chunk size               : 80 words
  Chunk overlap             : 20 words
  Total chunks              : 10

Embeddings
  Backend                  : sentence-transformers (all-MiniLM-L6-v2)
  Embedding dimension        : 384

Vector Store
  Engine                   : FAISS
  Index type                : IndexFlatIP (cosine via normalized inner product)
  Vectors stored             : 10

Generation
  Backend                  : google/flan-t5-base (transformers pipeline)

Experiments
  Hybrid search              : BM25 + vector similarity, weighted (alpha=0.5)
  Re-ranking                 : lexical-overlap re-scoring of top-5 candidates
  Chunking variants tested    : (40,10), (80,20), (150,30)

Validation
  Test questions run        

# 14. Key Learnings

- How retrieval and generation combine to ground a language model's answers in real, private documents.
- Why chunking strategy (size & overlap) directly affects retrieval accuracy and context completeness.
- How embeddings and a vector database (FAISS) enable fast semantic similarity search.
- How hybrid search (keyword & vector) and re-ranking can correct cases where pure semantic search misses exact terminology.
- How to design a pipeline that degrades gracefully (TF-IDF / extractive fallback) when a full model stack isn't available, instead of failing outright.

# Conclusion

We implement a complete Retrieval-Augmented Generation pipeline, document ingestion, chunking, embedding, vector storage, retrieval, and grounded answer generation, and validates it against a set of domain-specific test questions. The same designs (swap in `data\` with any PDF, resume, or research paper) generalizes to chatbots, knowledge assistants, enterprise search, and AI-powered documentation tools.